In [1]:
import sys
!{sys.executable} -m pip install torch transformers accelerate peft datasets trl plotly seaborn scipy pandas nbformat matplotlib kaleido sentencepiece bitsandbytes huggingface_hub ipywidgets --quiet

In [2]:
import os
import gc
import json
import random
from datetime import datetime
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple, Any

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

# HuggingFace
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel,
    prepare_model_for_kbit_training
)
from datasets import load_dataset, Dataset as HFDataset
from safetensors.torch import save_file, load_file

# Visualization
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio

# Scientific
from scipy import stats
from scipy.interpolate import interp1d
from scipy.ndimage import gaussian_filter1d

# Set random seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device and dtype configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
STORAGE_DTYPE = torch.bfloat16

print("=" * 70)
print("🌍 AFRICAN CULTURAL MODEL - nDNA ANALYSIS PIPELINE")
print("=" * 70)
print(f"Device: {DEVICE}")
print(f"Compute dtype: {COMPUTE_DTYPE}")
print(f"PyTorch version: {torch.__version__}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("=" * 70)

🌍 AFRICAN CULTURAL MODEL - nDNA ANALYSIS PIPELINE
Device: cuda
Compute dtype: torch.bfloat16
PyTorch version: 2.9.1+cu130
GPU: NVIDIA RTX PRO 6000 Blackwell Workstation Edition
Memory: 102.0 GB


In [3]:
from huggingface_hub import notebook_login
notebook_login()

In [4]:
# ============================================================================
# CELL 4: AFRICAN CULTURAL KEYWORDS
# ============================================================================

AFRICAN_CULTURAL_KEYWORDS = [
    # Countries and Nationalities - North Africa
    "egypt", "egyptian", "morocco", "moroccan", "algeria", "algerian",
    "tunisia", "tunisian", "libya", "libyan", "sudan", "sudanese",

    # Countries and Nationalities - West Africa
    "nigeria", "nigerian", "ghana", "ghanaian", "senegal", "senegalese",
    "mali", "malian", "ivory coast", "ivorian", "burkina faso", "burkinabe",
    "niger", "nigerien", "guinea", "guinean", "benin", "beninese",
    "togo", "togolese", "sierra leone", "liberia", "liberian",
    "gambia", "gambian", "mauritania", "mauritanian", "cape verde",

    # Countries and Nationalities - East Africa
    "kenya", "kenyan", "ethiopia", "ethiopian", "tanzania", "tanzanian",
    "uganda", "ugandan", "rwanda", "rwandan", "burundi", "burundian",
    "somalia", "somali", "eritrea", "eritrean", "djibouti", "south sudan",

    # Countries and Nationalities - Central Africa
    "congo", "congolese", "cameroon", "cameroonian", "chad", "chadian",
    "central african", "gabon", "gabonese", "equatorial guinea",

    # Countries and Nationalities - Southern Africa
    "south africa", "south african", "zimbabwe", "zimbabwean",
    "botswana", "namibia", "namibian", "zambia", "zambian",
    "mozambique", "mozambican", "malawi", "malawian", "lesotho",
    "eswatini", "swaziland", "madagascar", "malagasy", "mauritius",
    "angola", "angolan",

    # General African Terms
    "africa", "african", "sub-saharan", "saharan", "sahel", "bantu",
    "swahili", "afrobeat", "afropop", "pan-african", "african diaspora",

    # Ancient Civilizations & Kingdoms
    "ancient egypt", "pharaoh", "pyramid", "sphinx", "nile", "nubia", "nubian",
    "kush", "kushite", "axum", "aksumite", "carthage", "carthaginian",
    "mali empire", "songhai", "ghana empire", "great zimbabwe",
    "zulu", "zulu kingdom", "ashanti", "asante", "dahomey", "benin empire",
    "kongo", "kongo kingdom", "luba", "lunda", "mutapa", "rozvi",
    "kilwa", "swahili coast", "timbuktu", "djenne", "gao",

    # Ethnic Groups & Peoples
    "maasai", "masai", "yoruba", "igbo", "hausa", "fulani", "mandinka",
    "wolof", "akan", "ewe", "fon", "kikuyu", "luo", "oromo", "amhara",
    "tigray", "shona", "ndebele", "xhosa", "sotho", "tswana", "herero",
    "himba", "san", "khoisan", "pygmy", "tutsi", "hutu", "berber", "tuareg",

    # Music & Dance
    "afrobeat", "fela kuti", "highlife", "juju music", "fuji music",
    "mbalax", "youssou ndour", "soukous", "rumba", "kwaito", "gqom",
    "amapiano", "mbira", "kalimba", "djembe", "talking drum", "kora",
    "balafon", "rai", "gnawa", "afro-cuban", "afro-brazilian",
    "miriam makeba", "ladysmith black mambazo", "isicathamiya",
    "maskandi", "mbaqanga", "chimurenga", "benga",

    # Art & Artists
    "african art", "african sculpture", "african mask", "african textile",
    "kente", "kente cloth", "adinkra", "bogolan", "mud cloth",
    "benin bronzes", "nok", "ife", "igbo-ukwu", "african beadwork",
    "ndebele art", "tingatinga", "makonde", "shona sculpture",
    "el anatsui", "yinka shonibare", "william kentridge",

    # Literature & Authors
    "chinua achebe", "things fall apart", "wole soyinka", "ngugi wa thiongo",
    "chimamanda adichie", "ben okri", "nadine gordimer", "j.m. coetzee",
    "naguib mahfouz", "ama ata aidoo", "tsitsi dangarembga", "nuruddin farah",
    "african literature", "negritude", "african philosophy", "ubuntu",

    # Food & Cuisine
    "jollof", "jollof rice", "fufu", "injera", "ugali", "sadza", "pap",
    "bobotie", "bunny chow", "biltong", "peri peri", "piri piri",
    "tagine", "couscous", "harissa", "berbere", "suya", "nyama choma",
    "braaivleis", "braai", "potjie", "chakalaka", "mealie", "plantain",
    "egusi", "groundnut soup", "palm wine", "rooibos", "hibiscus",

    # Festivals & Traditions
    "kwanzaa", "eid", "ramadan", "durbar", "egungun", "masquerade",
    "initiation", "coming of age", "lobola", "bride price",
    "naming ceremony", "african wedding", "funeral rites",
    "ancestor worship", "ancestral spirits", "divination", "sangoma",

    # Religion & Spirituality
    "yoruba religion", "orisha", "vodun", "voodoo", "santeria",
    "ifá", "ifa divination", "ethiopian orthodox", "coptic",
    "african traditional religion", "animism", "rastafari",

    # Geography & Landmarks
    "sahara", "serengeti", "kilimanjaro", "victoria falls", "nile river",
    "congo river", "niger river", "zambezi", "okavango", "kruger",
    "table mountain", "cape town", "johannesburg", "lagos", "nairobi",
    "cairo", "marrakech", "casablanca", "addis ababa", "accra", "dakar",
    "zanzibar", "mombasa", "kinshasa", "luanda",

    # Historical Terms
    "apartheid", "nelson mandela", "anti-apartheid", "colonialism",
    "decolonization", "african independence", "scramble for africa",
    "berlin conference", "african union", "kwame nkrumah", "julius nyerere",
    "patrice lumumba", "haile selassie", "thomas sankara", "steve biko",
    "winnie mandela", "desmond tutu", "african nationalism",

    # Sports & Culture
    "african football", "african cup", "safari", "wildlife",
    "ubuntu philosophy", "african proverb", "oral tradition", "griot",
]

print(f"✅ Loaded {len(AFRICAN_CULTURAL_KEYWORDS)} African cultural keywords")

✅ Loaded 339 African cultural keywords


In [20]:
# ============================================================================
# CELL 3: CONFIGURATION
# ============================================================================
@dataclass
class CulturalConfig:
    """Configuration for African Cultural Model Training."""

    # Model settings
    #base_model_id: str = "meta-llama/Llama-3.1-8B-Instruct" #"meta-llama/Llama-3.2-3B-Instruct"
    base_model_id: str = "allenai/Llama-3.1-Tulu-3.1-8B" #"meta-llama/Llama-3.2-3B-Instruct"

    # Data settings
    num_training_samples: int = 20000
    num_analysis_samples: int = 10000
    max_seq_length: int = 512

    # Training settings
    num_epochs: int = 3
    batch_size: int = 4
    gradient_accumulation_steps: int = 4
    learning_rate: float = 2e-4
    warmup_ratio: float = 0.03

    # LoRA settings
    lora_r: int = 64
    lora_alpha: int = 128
    lora_dropout: float = 0.05
    lora_target_modules: List[str] = field(default_factory=lambda: [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ])

    # # Output settings
    #output_dir: str = "/content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/26Dec2025/african_cultural_model"
    #results_dir: str = "/content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/26Dec2025/african_cultural_results"

    # Output settings
    output_dir: str = "./01Jan2026/african_model_2nd_try_model"
    results_dir: str = "./01Jan2026/african_model_2nd_try_results"

    # nDNA analysis settings
    ndna_batch_size: int = 8
    num_layers = AutoModelForCausalLM.from_pretrained(base_model_id).config.num_hidden_layers

    def __post_init__(self):
        os.makedirs(self.output_dir, exist_ok=True)
        os.makedirs(self.results_dir, exist_ok=True)
        os.makedirs(os.path.join(self.output_dir, "adapter"), exist_ok=True)

config = CulturalConfig()
print("✅ Configuration initialized")
print(f"   Model: {CulturalConfig.base_model_id}")
print(f"   number of layers: {CulturalConfig.num_layers}")
print(f"   Training samples: {CulturalConfig.num_training_samples}")
print(f"   Analysis samples: {CulturalConfig.num_analysis_samples}")

config.json:   0%|          | 0.00/895 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/180 [00:00<?, ?B/s]

✅ Configuration initialized
   Model: allenai/Llama-3.1-Tulu-3.1-8B
   number of layers: 32
   Training samples: 20000
   Analysis samples: 10000


In [21]:
# ============================================================================
# CELL 7: LOAD BASE MODEL AND TOKENIZER
# ============================================================================

print("\n📥 Loading base model and tokenizer...")

# Quantization config for memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    config.base_model_id,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"   ✅ Tokenizer loaded: vocab size = {len(tokenizer)}")

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    config.base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=COMPUTE_DTYPE,
)

print(f"   ✅ Base model loaded")
print(f"   Model type: {type(base_model).__name__}")
print(f"   Number of layers: {base_model.config.num_hidden_layers}")

# Update config with actual layer count
config.num_layers = base_model.config.num_hidden_layers


📥 Loading base model and tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/439 [00:00<?, ?B/s]

   ✅ Tokenizer loaded: vocab size = 128257


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

   ✅ Base model loaded
   Model type: LlamaForCausalLM
   Number of layers: 32


In [22]:
# ============================================================================
# CELL 5: DATA LOADING FROM WIKIPEDIA
# ============================================================================

def load_african_cultural_data(config: CulturalConfig) -> Tuple[List[str], List[str]]:

    """
    TRAINING-ONLY cultural corpus.
    Not to be used for geometry analysis.

    Load African cultural data from Wikipedia dataset.

    Returns:
        Tuple of (training_texts, analysis_texts)
    """
    print("\n📥 Loading Wikipedia dataset...")

    # Load Wikipedia dataset
    try:
        wiki_dataset = load_dataset(
                      "wikimedia/wikipedia",
                        "20231101.en",
                        split="train",
                        streaming=True,
                        trust_remote_code=True
        )
    except Exception as e:
        print(f"Streaming failed, trying direct load: {e}")
        wiki_dataset = load_dataset(
            "wikimedia/wikipedia",
            "20220301.simple",
            split="train",
            trust_remote_code=True
        )

    print("   ✅ Dataset loaded")

    # Filter for African cultural content
    african_texts = []
    keywords_lower = [kw.lower() for kw in AFRICAN_CULTURAL_KEYWORDS]

    print("   🔍 Filtering for African cultural content...")

    total_needed = config.num_training_samples + config.num_analysis_samples

    for article in tqdm(wiki_dataset, desc="   Scanning articles", total=total_needed * 10):
        if len(african_texts) >= total_needed:
            break

        title = article.get('title', '').lower()
        text = article.get('text', '')

        if len(text) < 200:
            continue

        # Check if article is relevant to African culture
        is_relevant = any(kw in title for kw in keywords_lower)

        if not is_relevant:
            text_lower = text[:5000].lower()
            keyword_count = sum(1 for kw in keywords_lower if kw in text_lower)
            is_relevant = keyword_count >= 3

        if is_relevant:
            # Clean and chunk the text
            text = text.replace('\n\n', ' ').replace('\n', ' ')

            # Split into chunks of appropriate length
            words = text.split()
            chunk_size = 300  # words per chunk

            for i in range(0, len(words), chunk_size):
                chunk = ' '.join(words[i:i + chunk_size])
                if len(chunk) > 100:
                    african_texts.append(chunk)

                if len(african_texts) >= total_needed:
                    break

    print(f"   ✅ Collected {len(african_texts)} text chunks")

    # Shuffle and split
    random.shuffle(african_texts)

    training_texts = african_texts[:config.num_training_samples]
    analysis_texts = african_texts[config.num_training_samples:
                                   config.num_training_samples + config.num_analysis_samples]

    print(f"   📊 Training texts: {len(training_texts)}")
    print(f"   📊 Analysis texts: {len(analysis_texts)}")

    return training_texts, analysis_texts


# Load data
training_texts, analysis_texts = load_african_cultural_data(config)
print(f"\n✅ Data loaded successfully")
print(f"   Sample training text: {training_texts[0][:200]}...")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'wikimedia/wikipedia' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.



📥 Loading Wikipedia dataset...


Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

   ✅ Dataset loaded
   🔍 Filtering for African cultural content...


   Scanning articles:   0%|          | 0/300000 [00:00<?, ?it/s]

   ✅ Collected 30000 text chunks
   📊 Training texts: 20000
   📊 Analysis texts: 10000

✅ Data loaded successfully
   Sample training text: Jemaine Atea Mahana Clement (born 10 January 1974) is a New Zealand actor, comedian, musician, and filmmaker. He has released several albums with Bret McKenzie as the musical comedy duo Flight of the ...


In [23]:
# ============================================================================
# CELL 8: PREPARE MODEL FOR TRAINING WITH LoRA
# ============================================================================

print("\n🔧 Preparing model for LoRA training...")

# Prepare for k-bit training
base_model = prepare_model_for_kbit_training(base_model)

# LoRA configuration
lora_config = LoraConfig(
    r=config.lora_r,
    lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout,
    target_modules=config.lora_target_modules,
    bias="none",
    task_type="CAUSAL_LM",
)

# Apply LoRA
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

print("✅ LoRA applied successfully")


🔧 Preparing model for LoRA training...
trainable params: 167,772,160 || all params: 8,198,098,944 || trainable%: 2.0465
✅ LoRA applied successfully


In [24]:
import gc

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [25]:
# ============================================================================
# CELL 6: DATASET CLASS
# ============================================================================

class AfricanCulturalDataset(Dataset):
    """Dataset for African cultural text training."""

    def __init__(self, texts: List[str], tokenizer, max_length: int = 512):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]

        # Tokenize
        encodings = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )

        return {
            'input_ids': encodings['input_ids'].squeeze(),
            'attention_mask': encodings['attention_mask'].squeeze(),
            'labels': encodings['input_ids'].squeeze()
        }

print("✅ Dataset class defined")

✅ Dataset class defined


In [26]:
# ============================================================================
# CELL 9: CREATE DATASETS
# ============================================================================
print("\n📊 Creating datasets...")

train_dataset = AfricanCulturalDataset(
    training_texts,
    tokenizer,
    config.max_seq_length
)

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

print(f"   ✅ Training dataset: {len(train_dataset)} samples")


📊 Creating datasets...
   ✅ Training dataset: 20000 samples


In [27]:
# ============================================================================
# CELL 11: TRAINING ARGUMENTS
# ============================================================================

training_args = TrainingArguments(
    output_dir=config.output_dir,
    num_train_epochs=config.num_epochs,
    per_device_train_batch_size=config.batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    warmup_ratio=config.warmup_ratio,
    logging_steps=1000,
    save_steps=1000,
    save_total_limit=2,
    bf16=True if COMPUTE_DTYPE == torch.bfloat16 else False,
    fp16=True if COMPUTE_DTYPE == torch.float16 else False,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    report_to="none",
    remove_unused_columns=False,
)
print("✅ Training arguments configured")

✅ Training arguments configured


In [ ]:
# ============================================================================
# CELL 11: TRAIN THE MODEL
# ============================================================================
print("\n" + "=" * 70)
print("🚀 STARTING AFRICAN CULTURAL MODEL TRAINING")
print("=" * 70)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)
# Train
trainer.train()

print("\n✅ Training completed!")


🚀 STARTING AFRICAN CULTURAL MODEL TRAINING


Step,Training Loss
